# TinyYOLO Experiment Suite — Notebook 6/7
## Quantization Validation (Table VII)

**What:** PTQ and QAT accuracy retention on the best VOC checkpoint.

**GPU Time:** ~4h T4

**Prerequisite:** Run Notebook 01 first to generate VOC checkpoints.

In [ ]:
import os, sys, socket, shutil
from pathlib import Path
REPO = 'https://github.com/ShMazumder/tinyYOLO.git'
try: socket.create_connection(('github.com', 80), timeout=5)
except: print('❌ No internet!'); sys.exit(1)
if Path('tinyYOLO').exists(): shutil.rmtree('tinyYOLO')
!git clone {REPO} --depth 1
%cd tinyYOLO
!pip install -e . -q 2>&1 | tail -1
!pip install tqdm timm psutil -q 2>&1 | tail -1
import torch
if torch.cuda.is_available(): print(f'🔥 GPU: {torch.cuda.get_device_name(0)}')
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')" 2>&1 | tail -3
print('✅ Setup complete!')

In [ ]:
import glob

# Find best checkpoints from Notebook 01
q_weights = sorted(glob.glob('experiments/results/voc-q-416-seed*/best.pt'))
s_weights = sorted(glob.glob('experiments/results/voc-s-416-seed*/best.pt'))

print(f'Found {len(q_weights)} quantized checkpoints')
print(f'Found {len(s_weights)} standard checkpoints')

if not q_weights:
    print('\n⚠️  No checkpoints found! Run Notebook 01 first.')
    print('    Or upload a best.pt to experiments/results/voc-q-416-seed42/')

# Use first seed checkpoint for quantization experiments
Q_BEST = q_weights[0] if q_weights else None
S_BEST = s_weights[0] if s_weights else None
print(f'\nUsing: Q={Q_BEST}, S={S_BEST}')

In [ ]:
# PTQ on quantized variant (should retain well due to ReLU6+ECA)
if Q_BEST:
    !python scripts/quantize.py --mode ptq --variant quantized \
        --weights {Q_BEST} --data voc.yaml --imgsz 416 \
        --batch 16 --n-calib 500 --backend qnnpack
else:
    print('⏭️  Skipped — no quantized checkpoint')

In [ ]:
# QAT on quantized variant (10 epochs fine-tuning)
if Q_BEST:
    !python scripts/quantize.py --mode qat --variant quantized \
        --weights {Q_BEST} --data voc.yaml --imgsz 416 \
        --batch 16 --epochs 10 --lr 1e-4 --backend qnnpack
else:
    print('⏭️  Skipped — no quantized checkpoint')

In [ ]:
# PTQ on standard variant (expected larger degradation due to SiLU)
if S_BEST:
    !python scripts/quantize.py --mode ptq --variant standard \
        --weights {S_BEST} --data voc.yaml --imgsz 416 \
        --batch 16 --n-calib 500 --backend qnnpack
else:
    print('⏭️  Skipped — no standard checkpoint')

In [ ]:
# QAT on standard variant
if S_BEST:
    !python scripts/quantize.py --mode qat --variant standard \
        --weights {S_BEST} --data voc.yaml --imgsz 416 \
        --batch 16 --epochs 10 --lr 1e-4 --backend qnnpack
else:
    print('⏭️  Skipped — no standard checkpoint')

In [ ]:
import json, glob
print('='*70)
print('QUANTIZATION RESULTS')
print('='*70)
for qdir in sorted(glob.glob('experiments/results/voc-*/quantized')):
    print(f'\n📁 {qdir}')
    for sf in sorted(Path(qdir).glob('quantization_stats_*.json')):
        with open(sf) as f: stats = json.load(f)
        print(f'  {sf.name}:')
        for k, v in stats.items():
            if k != 'history': print(f'    {k}: {v}')
print('='*70)